# Arene des Algos
Pipeline ML complet : classification supervisée, clustering non-supervisé, scaling, visualisations.

## Phase 1 : Charger et explorer
Dataset : `breast_cancer` (sklearn) : 569 patients, 30 mesures, tumeur bénigne ou maligne.

In [1]:
from sklearn.datasets import load_breast_cancer
import numpy as np


def explorer_dataset(X=None, y=None, noms_classes=None):
    """Charge le dataset et affiche ses caractéristiques de base.
    Affiche : nombre de lignes, nombre de colonnes,
    les classes possibles et leur répartition (équilibrée ou non ?).
    Si X et y sont None, charge breast_cancer par défaut.
    """
    if X is None or y is None:
        dataset = load_breast_cancer()
        X, y = dataset.data, dataset.target
        noms_classes = dataset.target_names

    print(f"- Lignes, colonnes : {X.shape}")
    total = len(y)
    for i in np.unique(y):
        nom = noms_classes[i] if noms_classes is not None and i < len(noms_classes) else str(i)
        count = int(np.sum(y == i))
        pct = count / total * 100
        print(f"- Classe {i} ({nom}) : {count} cas  ({pct:.1f}%)")


explorer_dataset()

- Lignes, colonnes : (569, 30)
- Classe 0 (malignant) : 212 cas  (37.3%)
- Classe 1 (benign) : 357 cas  (62.7%)


### Checkpoints qualité
Trois cas testés pour vérifier qu'`explorer_dataset` ne ment pas :
- **Cas normal** : le dataset complet
- **Cas limite** : une seule classe (y == 0), le décompte doit rester honnête
- **Cas adversarial** : répartition en pourcentage avec un déséquilibre à 95% 

In [2]:
dataset = load_breast_cancer()
X_full, y_full = dataset.data, dataset.target
noms = dataset.target_names

# Cas normal
print("Cas normal (dataset complet)")
explorer_dataset()

# Cas limite : une seule classe
print("\nCas limite (seulement classe 0 — maligne)")
y_full == 0
mask = y_full
explorer_dataset(X_full[mask], y_full[mask], noms)

# Cas adversarial : déséquilibre artificiel à ~95%
print("\nCas adversarial (déséquilibre 95/5)")
n_minoritaire = int(len(y_full) * 0.05)
idx_minoritaire = np.where(y_full == 0)[0]
idx_majoritaire = np.where(y_full == 1)[0][:n_minoritaire]
idx = np.concatenate([idx_minoritaire, idx_majoritaire])
explorer_dataset(X_full[idx], y_full[idx], noms)

Cas normal (dataset complet)
- Lignes, colonnes : (569, 30)
- Classe 0 (malignant) : 212 cas  (37.3%)
- Classe 1 (benign) : 357 cas  (62.7%)

Cas limite (seulement classe 0 — maligne)
- Lignes, colonnes : (569, 30)
- Classe 0 (malignant) : 569 cas  (100.0%)

Cas adversarial (déséquilibre 95/5)
- Lignes, colonnes : (240, 30)
- Classe 0 (malignant) : 212 cas  (88.3%)
- Classe 1 (benign) : 28 cas  (11.7%)


## Phase 2 : Pipeline supervisé complet

Split train/test, entraînement, prédiction, mesure encapsulé dans une fonction réutilisable.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier


def entrainer_et_evaluer(modele, X_train, X_test, y_train, y_test):
    """Entraine le modele, predit sur le test, renvoie l'accuracy.
    Renvoie un float entre 0 et 1.
    """
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)
    return accuracy_score(y_test, y_pred)


# Préparation du split
# je fixe random_state pour que le résultat soit reproductible d'une exécution à l'autre
X, y = dataset.data, dataset.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Test rapide avec un arbre de décision
# idem ici pour random_state
acc = entrainer_et_evaluer(DecisionTreeClassifier(random_state=42), X_train, X_test, y_train, y_test)
print(f"Accuracy arbre de décision : {acc:.1%}")

Accuracy arbre de décision : 94.7%


## Phase 3 : L'Arène: premier classement

Plusieurs algorithmes s'affrontent sur le **même** découpage train/test.
Classement trié par accuracy décroissante.

In [4]:
import warnings
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier


def arene(X_train, X_test, y_train, y_test):
    """Entraine plusieurs modeles, renvoie un classement trié par accuracy décroissante.
    Affiche un tableau lisible : rang, nom de l'algo, accuracy.
    """
    modeles = {
        "Arbre de décision":    DecisionTreeClassifier(random_state=42),
        "Régression logistique": LogisticRegression(max_iter=10000, random_state=42),
        "KNN":                   KNeighborsClassifier(),
    }

    resultats = {}
    for nom, modele in modeles.items():
        with warnings.catch_warnings(): # pour capturer les warnings de cvg de LogisticRegression
            warnings.simplefilter("ignore")
            acc = entrainer_et_evaluer(modele, X_train, X_test, y_train, y_test)
        resultats[nom] = acc

    # Tri par accuracy décroissante
    classement = sorted(resultats.items(), key=lambda x: x[1], reverse=True)

    print(f"{'Rang':<5} {'Algorithme':<25} {'Accuracy':>8}")
    print("-" * 42)
    for rang, (nom, acc) in enumerate(classement, start=1):
        print(f"{rang:<5} {nom:<25} {acc:>7.1%}")

    return classement


print("- Arène : breast_cancer\n")
classement_breast = arene(X_train, X_test, y_train, y_test)

- Arène : breast_cancer

Rang  Algorithme                Accuracy
------------------------------------------
1     Régression logistique       95.6%
2     KNN                         95.6%
3     Arbre de décision           94.7%


## Phase 4 : Bascule non-supervisé clustering aveugle

On cache les étiquettes. KMeans doit trouver 2 groupes tout seul, sans jamais voir `y`.
Puis on compare : les groupes trouvés correspondent-ils aux vraies classes ?

In [5]:
from sklearn.cluster import KMeans


def clustering_aveugle(X):
    """Regroupe les données en 2 clusters sans les étiquettes.
    Renvoie les labels de cluster attribués à chaque point.
    """
    # fit_predict fait l'entraînement ET la prédiction en une seule passe
    kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
    return kmeans.fit_predict(X)


# KMeans ne voit que X, pas y
labels_clusters = clustering_aveugle(X)

# Comparaison avec les vraies classes
# on calcule les deux alignements possibles et on garde le meilleur 
# vu que KMeans peut inverser les numéros de clusters
accord_direct = np.mean(labels_clusters == y)
accord_inverse = np.mean(labels_clusters != y)  # clusters inversés
accord = max(accord_direct, accord_inverse)

print(f"Accord clustering / vraies classes : {accord:.1%}")
print()
print("Répartition des clusters trouvés :")
for c in [0, 1]:
    print(f"  Cluster {c} : {np.sum(labels_clusters == c)} points")

Accord clustering / vraies classes : 85.4%

Répartition des clusters trouvés :
  Cluster 0 : 131 points
  Cluster 1 : 438 points


## Phase 5 : Changer de terrain — dataset Wine

Même Arène, nouveau dataset : `load_wine` — 178 vins, 13 mesures, **3 classes** (multi-classe).
Les fonctions `explorer_dataset` et `arene` ne doivent pas changer d'une ligne.

In [ ]:
from sklearn.datasets import load_wine

dataset_wine = load_wine()
X_wine, y_wine = dataset_wine.data, dataset_wine.target

# Exploration — même fonction, 3 classes cette fois
print("=== Exploration wine ===")
explorer_dataset(X_wine, y_wine, dataset_wine.target_names)

# Split
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_wine, y_wine, test_size=0.2, random_state=42
)

# Arène — même fonction, encaisse les 3 classes sans modification
print("\n=== Arène — wine ===\n")
classement_wine = arene(X_train_w, X_test_w, y_train_w, y_test_w)